# PPE 3 lớp (Person · Helmet · Safety-Vest) — dataset Construction Site Safety (Roboflow) · YOLO11

Dataset: [Construction Site Safety Image Dataset](https://www.kaggle.com/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow)
(Roboflow Universe, giấy phép **CC BY 4.0** — cần ghi nguồn khi dùng).

**Mục tiêu đồ án D08:** model chỉ phát hiện đối tượng; kết luận vi phạm do Rule Engine (Tracking + Association + thời gian).

### Đổi nhãn

| id gốc | Nhãn gốc | → | id mới | Nhãn mới |
|---|---|---|---|---|
| 5 | Person | → | **0** | Person |
| 0 | Hardhat | → | **1** | Helmet |
| 7 | Safety Vest | → | **2** | Safety-Vest |
| 1, 3 | Mask, NO-Mask | → | | **xóa** |
| 2, 4 | NO-Hardhat, NO-Safety Vest | → | | **xóa** (đầu/thân không PPE trở thành nền) |
| 6, 8, 9 | Safety Cone, machinery, vehicle | → | | **xóa** (ngoài phạm vi hiện tại) |

> Nếu sau này đồ án chuyển sang hướng có lớp phủ định, chỉ cần thêm `2: 3, 4: 4` vào `CFG.REMAP` và mở rộng `CFG.NEW_CLASSES` + `COLORS`.

### Điều cần biết về dataset này
- Bản Roboflow v27 có 2801 ảnh nhưng **tập train đã được augment sẵn** (train 2605 ảnh so với 521 ảnh gốc ở bản raw; valid 114, test 82).
  Số ảnh *độc lập* chỉ khoảng 717 → đừng báo cáo "2.8k ảnh". Notebook này in số ảnh nguồn thật ở mục 3.
- Valid/test rất nhỏ (114 / 82 ảnh) → mAP dao động mạnh, nhất là Safety-Vest.

### Xử lý dữ liệu
- Dùng YOLO11s và cùng siêu tham số với notebook 01 → khác biệt kết quả đến từ dữ liệu, không phải từ model.
- Đánh giá cả valid lẫn test, từng lớp; kiểm tra rò rỉ theo ảnh nguồn Roboflow.
- Giữ ≤ 10% ảnh nền; đầu ra gồm kiểm toán nhãn trước/sau, ảnh kiểm tra nhãn, đo ms/ảnh, xuất `.onnx` + `metadata.json`.

In [ ]:
!pip install -q -U ultralytics
import torch, ultralytics
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("ultralytics", ultralytics.__version__)

## 1. Cấu hình

In [ ]:
import os, glob, json, random, shutil, time, yaml
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image


class CFG:
    SEED  = 42
    DEBUG = False

    NEW_CLASSES = {0: "Person", 1: "Helmet", 2: "Safety-Vest"}

    SRC_NAMES = {0: "Hardhat", 1: "Mask", 2: "NO-Hardhat", 3: "NO-Mask", 4: "NO-Safety Vest",
                 5: "Person", 6: "Safety Cone", 7: "Safety Vest", 8: "machinery", 9: "vehicle"}
    REMAP = {5: 0, 0: 1, 7: 2}          # lớp không có trong dict sẽ bị XÓA

    SRC_ROOT = os.environ.get("PPE_SRC")
    SRC_CANDIDATES = [
        "/kaggle/input/construction-site-safety-image-dataset-roboflow/css-data",
        "/kaggle/input/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow/css-data",
    ]

    BG_RATIO = 0.10        # ảnh nền tối đa = 10% số ảnh có box (áp dụng riêng từng tập)

    BASE_MODEL = "yolo11s.pt"
    IMGSZ      = 640
    EPOCHS     = 3 if DEBUG else 100
    PATIENCE   = 5 if DEBUG else 25
    BATCH      = 16

    WORK     = os.environ.get("PPE_WORK") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./work"))
    DATA_DIR = f"{WORK}/ppe3_css"
    RUNS_DIR = f"{WORK}/runs"
    OUT_DIR  = f"{WORK}/Artifacts"
    EXP_NAME = "css_3cls_yolo11s"

def find_src_root():
    for p in [CFG.SRC_ROOT] + CFG.SRC_CANDIDATES:
        if p and os.path.isdir(os.path.join(p, "train")):
            return p
    raise FileNotFoundError("Không tìm thấy dataset. Hãy gán CFG.SRC_ROOT = '<thư mục css-data chứa train/ valid/ test/>'")

CFG.SRC_ROOT = find_src_root()
print("Dataset nguồn:", CFG.SRC_ROOT, "\nThư mục làm việc:", CFG.WORK)

## 2. Kiểm tra thứ tự lớp gốc, đọc dữ liệu, đổi nhãn

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
NAMES   = CFG.NEW_CLASSES
COLORS  = {0: (60, 120, 255), 1: (0, 200, 80), 2: (255, 170, 0)}   # RGB để vẽ

def remap_label(lbl_path):
    """Đọc 1 file nhãn YOLO -> giữ + đổi id các lớp trong CFG.REMAP, bỏ phần còn lại."""
    out, kept, dropped, seen, bad = [], Counter(), Counter(), Counter(), 0
    with open(lbl_path) as f:
        for line in f:
            p = line.split()
            if not p:
                continue
            try:
                cid = int(float(p[0]))
                x, y, w, h = map(float, p[1:5])
            except (ValueError, IndexError):
                bad += 1; continue
            if len(p) != 5:                      # nhãn polygon/segment -> không dùng
                bad += 1; continue
            seen[cid] += 1
            if cid not in CFG.REMAP:
                dropped[cid] += 1; continue
            x1, y1 = max(0.0, x - w / 2), max(0.0, y - h / 2)    # cắt box về trong ảnh
            x2, y2 = min(1.0, x + w / 2), min(1.0, y + h / 2)
            w, h = x2 - x1, y2 - y1
            if w <= 0.002 or h <= 0.002:
                bad += 1; continue
            nid = CFG.REMAP[cid]
            out.append(f"{nid} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} {w:.6f} {h:.6f}")
            kept[nid] += 1
    return dict(lines=out, kept=dict(kept), dropped=dict(dropped), seen=dict(seen), bad=bad)

def build_table(pairs):
    rows = []
    for p in pairs:
        r = remap_label(p["lbl"])
        rows.append({**p, **r, "n_kept": sum(r["kept"].values())})
    return pd.DataFrame(rows)

def audit(df, title="Kiểm toán nhãn"):
    """So sánh số instance TRƯỚC (nhãn gốc) và SAU (nhãn mới)."""
    seen, kept = Counter(), Counter()
    for d in df["seen"]:  seen.update(d)
    for d in df["kept"]:  kept.update(d)
    rows = []
    for sid, name in CFG.SRC_NAMES.items():
        rows.append(dict(id_goc=sid, ten_goc=name, instance_goc=seen.get(sid, 0),
                         ket_qua=("-> " + NAMES[CFG.REMAP[sid]]) if sid in CFG.REMAP else "XÓA"))
    print(f"== {title} ==")
    display(pd.DataFrame(rows)) if "display" in globals() else print(pd.DataFrame(rows).to_string(index=False))
    print("Sau khi đổi nhãn:", {NAMES[c]: kept.get(c, 0) for c in NAMES})
    print(f"Ảnh có >=1 box giữ lại: {(df.n_kept > 0).sum()} | ảnh trống sau khi lọc: {(df.n_kept == 0).sum()} | box lỗi bỏ qua: {int(df.bad.sum())}")

def select_images(df, seed):
    """Giữ mọi ảnh có box; ảnh trống (background) chỉ giữ tối đa BG_RATIO * số ảnh có box."""
    pos, bg = df[df.n_kept > 0], df[df.n_kept == 0]
    n_bg = min(len(bg), int(round(len(pos) * CFG.BG_RATIO)))
    bg = bg.sample(n=n_bg, random_state=seed) if n_bg else bg.iloc[0:0]
    return pd.concat([pos, bg]).reset_index(drop=True)

def reset_dir(path):
    if os.path.isdir(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def write_split(df, split):
    di, dl = Path(CFG.DATA_DIR) / split / "images", Path(CFG.DATA_DIR) / split / "labels"
    di.mkdir(parents=True, exist_ok=True); dl.mkdir(parents=True, exist_ok=True)
    for r in df.itertuples():
        ext = Path(r.img).suffix.lower()
        shutil.copy(r.img, di / f"{r.uid}{ext}")
        (dl / f"{r.uid}.txt").write_text("\n".join(r.lines) + ("\n" if r.lines else ""))

def write_yaml():
    cfg = {"path": CFG.DATA_DIR, "train": "train/images", "val": "valid/images",
           "test": "test/images", "names": {int(k): v for k, v in NAMES.items()}}
    path = f"{CFG.DATA_DIR}/dataset.yaml"
    with open(path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
    return path

def split_stats():
    rows = []
    for sp in ["train", "valid", "test"]:
        cnt, n_img, n_bg = Counter(), 0, 0
        for lf in glob.glob(f"{CFG.DATA_DIR}/{sp}/labels/*.txt"):
            n_img += 1
            ids = [int(l.split()[0]) for l in open(lf) if l.strip()]
            cnt.update(ids); n_bg += (len(ids) == 0)
        rows.append({"split": sp, "ảnh": n_img, "ảnh trống": n_bg, **{NAMES[c]: cnt.get(c, 0) for c in NAMES}})
    return pd.DataFrame(rows)

def draw_gt(img_path, lbl_path):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    for line in open(lbl_path):
        if not line.strip():
            continue
        c, x, y, w, h = line.split(); c = int(c)
        x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
        p1, p2 = (int(x - w / 2), int(y - h / 2)), (int(x + w / 2), int(y + h / 2))
        cv2.rectangle(img, p1, p2, COLORS[c], 2)
        cv2.putText(img, NAMES[c], (p1[0], max(12, p1[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLORS[c], 2)
    return img

def show_samples(split="train", n=8, seed=0):
    files = sorted(glob.glob(f"{CFG.DATA_DIR}/{split}/images/*"))
    files = [f for f in files if os.path.getsize(f"{CFG.DATA_DIR}/{split}/labels/{Path(f).stem}.txt") > 0]
    random.Random(seed).shuffle(files)
    files = files[:n]
    cols = 4; rows = (len(files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4)); axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, f in zip(axes, files):
        ax.imshow(draw_gt(f, f"{CFG.DATA_DIR}/{split}/labels/{Path(f).stem}.txt"))
    plt.tight_layout(); plt.show()


# --- chống đổi nhầm nhãn âm thầm: so thứ tự lớp trong data.yaml của dataset với SRC_NAMES ---
norm = lambda s: str(s).lower().replace(" ", "").replace("-", "").replace("_", "")
yml = Path(CFG.SRC_ROOT) / "data.yaml"
if yml.exists():
    names = yaml.safe_load(open(yml)).get("names")
    names = list(names.values()) if isinstance(names, dict) else list(names)
    expect = [CFG.SRC_NAMES[i] for i in sorted(CFG.SRC_NAMES)]
    if [norm(x) for x in names] != [norm(x) for x in expect]:
        raise ValueError(f"Thứ tự lớp trong data.yaml KHÁC SRC_NAMES:\n  yaml: {names}\n  cfg : {expect}\nSửa CFG.SRC_NAMES/REMAP rồi chạy lại.")
    print("data.yaml khớp SRC_NAMES ✓")
else:
    print("Không có data.yaml để đối chiếu -> hãy xem kỹ ảnh kiểm tra ở mục 4.")

def collect_pairs(root):
    pairs = []
    for sp_src, sp in [("train", "train"), ("valid", "valid"), ("val", "valid"), ("test", "test")]:
        idir, ldir = Path(root) / sp_src / "images", Path(root) / sp_src / "labels"
        if not idir.is_dir():
            continue
        imgs = {p.stem: p for p in idir.iterdir() if p.suffix.lower() in IMG_EXT}
        lbls = {p.stem: p for p in ldir.glob("*.txt")}
        for stem in sorted(set(imgs) & set(lbls)):
            pairs.append(dict(uid=stem, img=str(imgs[stem]), lbl=str(lbls[stem]), split=sp,
                              src_id=stem.split(".rf.")[0]))      # phần trước ".rf." = ảnh gốc trước khi Roboflow augment
    return pairs

df_all = build_table(collect_pairs(CFG.SRC_ROOT))
audit(df_all, "Construction Site Safety (Roboflow)")

## 3. Kiểm tra rò rỉ theo ảnh nguồn, chọn ảnh, ghi dữ liệu

In [ ]:
SPLITS = ["train", "valid", "test"]
src = {sp: set(df_all[df_all.split == sp].src_id) for sp in SPLITS}
for sp in SPLITS:
    n = int((df_all.split == sp).sum())
    print(f"{sp:>5}: {n:5d} ảnh | {len(src[sp]):4d} ảnh nguồn độc lập | hệ số augment ≈ {n / max(len(src[sp]), 1):.1f}x")

leaked = (src["valid"] | src["test"]) & src["train"]
if leaked:                                    # ảnh nguồn xuất hiện cả ở train lẫn valid/test -> bỏ khỏi train
    mask = (df_all.split == "train") & df_all.src_id.isin(leaked)
    print(f"⚠ {len(leaked)} ảnh nguồn rò rỉ -> loại {int(mask.sum())} ảnh khỏi train")
    df_all = df_all[~mask]
else:
    print("Không có ảnh nguồn nào xuất hiện ở nhiều tập ✓")
print("Chồng lấn valid ∩ test:", len(src["valid"] & src["test"]))

parts = [select_images(df_all[df_all.split == sp], CFG.SEED).assign(split=sp) for sp in SPLITS]
reset_dir(CFG.DATA_DIR)
for sp, part in zip(SPLITS, parts):
    write_split(part, sp)
YAML_PATH = write_yaml()
stats = split_stats(); print(); print(stats.to_string(index=False)); print("\nYAML:", YAML_PATH)

## 4. Kiểm tra bằng mắt: nhãn đã đổi có đúng không?
Person = xanh dương, Helmet = xanh lá, Safety-Vest = cam. Nếu màu/tên sai → sửa `CFG.REMAP` rồi chạy lại từ mục 1.

In [ ]:
stats.set_index("split")[list(NAMES.values())].plot(kind="bar", figsize=(8, 4), rot=0, title="Số instance theo lớp")
plt.tight_layout(); plt.show()
show_samples("train", n=8)

## 5. Huấn luyện YOLO11

In [ ]:
os.environ["WANDB_DISABLED"] = "true"
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"
YAML_PATH = f"{CFG.DATA_DIR}/dataset.yaml"
random.seed(CFG.SEED); np.random.seed(CFG.SEED)

model = YOLO(CFG.BASE_MODEL)
model.train(
    data=YAML_PATH, epochs=CFG.EPOCHS, patience=CFG.PATIENCE, batch=CFG.BATCH,
    imgsz=CFG.IMGSZ, device=DEVICE, optimizer="auto", cos_lr=True, seed=CFG.SEED,
    fraction=0.1 if CFG.DEBUG else 1.0, project=CFG.RUNS_DIR, name=CFG.EXP_NAME,
    exist_ok=True, plots=True, workers=4,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
print("best.pt:", BEST_PT)

## 6. Đánh giá (valid + test, từng lớp)

In [ ]:
# --- đường cong huấn luyện ---
df_res = pd.read_csv(RUN_DIR / "results.csv"); df_res.columns = df_res.columns.str.strip()
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for a, cols, t in zip(ax, [("train/box_loss", "val/box_loss"), ("train/cls_loss", "val/cls_loss"),
                           ("metrics/mAP50(B)", "metrics/mAP50-95(B)")], ["Box loss", "Cls loss", "mAP"]):
    for c in cols: a.plot(df_res["epoch"], df_res[c], label=c)
    a.set_title(t); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

# --- đánh giá từng tập, từng lớp ---
best = YOLO(str(BEST_PT))

def eval_split(split):
    m = best.val(data=YAML_PATH, split=split, imgsz=CFG.IMGSZ, batch=16, device=DEVICE,
                 plots=(split == "test"), project=CFG.RUNS_DIR, name=f"eval_{split}",
                 exist_ok=True, verbose=False)
    rows = [dict(Lop=NAMES[int(c)], P=m.box.p[i], R=m.box.r[i], mAP50=m.box.ap50[i], mAP50_95=m.box.ap[i])
            for i, c in enumerate(m.box.ap_class_index)]
    rows.append(dict(Lop="ALL", P=m.box.mp, R=m.box.mr, mAP50=m.box.map50, mAP50_95=m.box.map))
    return pd.DataFrame(rows).round(3)

res_val, res_test = eval_split("valid"), eval_split("test")
print("== VALID =="); print(res_val.to_string(index=False))
print("\n== TEST (chỉ dùng để báo cáo cuối) =="); print(res_test.to_string(index=False))

## 7. Ảnh dự đoán mẫu và tốc độ suy luận

In [ ]:
test_imgs = sorted(glob.glob(f"{CFG.DATA_DIR}/test/images/*"))
sample = random.Random(1).sample(test_imgs, min(8, len(test_imgs)))
preds = best.predict(sample, conf=0.35, imgsz=CFG.IMGSZ, device=DEVICE, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(20, 10)); axes = axes.reshape(-1)
for ax in axes: ax.axis("off")
for ax, r in zip(axes, preds):
    ax.imshow(r.plot()[..., ::-1])
    cnt = Counter(NAMES[int(c)] for c in r.boxes.cls.tolist())
    ax.set_title(", ".join(f"{k}:{v}" for k, v in cnt.items()) or "không phát hiện")
plt.tight_layout(); plt.show()

# --- tốc độ suy luận (1 ảnh/lần) trên chính máy đang chạy ---
bench = test_imgs[:50]
best.predict(bench[:3], imgsz=CFG.IMGSZ, device=DEVICE, verbose=False)      # warm-up
t0 = time.perf_counter()
for p in bench:
    best.predict(p, imgsz=CFG.IMGSZ, device=DEVICE, conf=0.25, verbose=False)
dt = (time.perf_counter() - t0) / len(bench)
print(f"{dt * 1000:.1f} ms/ảnh  ≈ {1 / dt:.1f} FPS  | thiết bị: {DEVICE}")

## 8. Lưu model (.pt, .onnx) và metadata

In [ ]:
os.makedirs(CFG.OUT_DIR, exist_ok=True)
shutil.copy(BEST_PT, f"{CFG.OUT_DIR}/{CFG.EXP_NAME}.pt")
try:
    onnx = best.export(format="onnx", imgsz=CFG.IMGSZ, simplify=True)
except Exception as e:
    print("simplify lỗi, xuất lại không simplify:", e)
    onnx = best.export(format="onnx", imgsz=CFG.IMGSZ, simplify=False)
shutil.copy(onnx, f"{CFG.OUT_DIR}/{CFG.EXP_NAME}.onnx")

meta = {
    "experiment": CFG.EXP_NAME, "base_model": CFG.BASE_MODEL, "imgsz": CFG.IMGSZ,
    "epochs": CFG.EPOCHS, "batch": CFG.BATCH, "seed": CFG.SEED,
    "classes": {int(k): v for k, v in NAMES.items()},          # AI Vision Service đọc file này
    "source_remap": {CFG.SRC_NAMES[k]: NAMES[v] for k, v in CFG.REMAP.items()},
    "dataset_stats": split_stats().to_dict(orient="records"),
    "valid": res_val.to_dict(orient="records"), "test": res_test.to_dict(orient="records"),
    "ms_per_image": round(dt * 1000, 2), "device": str(DEVICE),
}
with open(f"{CFG.OUT_DIR}/{CFG.EXP_NAME}_metadata.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Đã lưu vào", CFG.OUT_DIR, "->", os.listdir(CFG.OUT_DIR))

## Bước tiếp theo (phần còn thiếu để bạn tự bổ sung)

1. **Safety-Vest là lớp yếu dữ liệu nhất** → bổ sung từ dataset khác (CHV, Pictor-PPE, SH17...). Nhớ đổi nhãn về đúng schema `0 Person / 1 Helmet / 2 Safety-Vest` bằng cách sửa `CFG.SRC_NAMES` + `CFG.REMAP` rồi chạy lại.
2. **Gộp nhiều dataset:** vì cùng schema nên có thể copy các thư mục `ppe3_*` vào một chỗ, nhưng phải **chia lại theo nhóm ảnh gần giống nhau** (hàm `dup_groups` + `group_split`) để không rò rỉ train/test.
3. **Bộ test cố định cho báo cáo:** chọn một tập ảnh/khung hình từ video công trường (ưu tiên khác nguồn với train) và không dùng để chọn model.
4. **Chỉ số của đề tài không dừng ở mAP:** sau bước này cần Tracking + Association + Rule Engine rồi mới đo Violation Precision/Recall, False Alert Rate, Duplicate Alert Rate trên video có nhãn sự kiện.